# Session 4: Introduction to MCP (Model Context Protocol)
**Duration:** 1 Hour | **Day 1** | **Hands-On Workshop**

---

## Learning Objectives
By the end of this session, participants will:
1. Understand what MCP is and why it matters for AI applications
2. Grasp the MCP architecture (Host, Client, Server)
3. Understand MCP primitives: Tools, Resources, and Prompts
4. Build a custom MCP server with callable tools
5. Test MCP tools locally

---

## Prerequisites
- Python 3.10+
- Packages: `pip install mcp httpx`
- Understanding of APIs and JSON

---
## Part 1: What is MCP and Why It Matters (Theory - 15 min)

### 1.1 The Problem MCP Solves

Without MCP, every AI application needs custom integration code:

```
BEFORE MCP (N x M integration problem):

LLM App 1 ---custom code---> Database
LLM App 1 ---custom code---> File System
LLM App 1 ---custom code---> API Service
LLM App 2 ---custom code---> Database
LLM App 2 ---custom code---> File System
LLM App 2 ---custom code---> API Service
  ... (every app needs to integrate with every tool separately)


WITH MCP (standardized protocol):

LLM App 1 ---MCP---> MCP Server (Database)
LLM App 2 ---MCP---> MCP Server (Database)   // Same server, no custom code
LLM App 1 ---MCP---> MCP Server (File System)
LLM App 2 ---MCP---> MCP Server (File System) // Same server, no custom code
```

**MCP is like USB for AI applications** -- a universal standard for connecting LLMs to tools and data.

### 1.2 MCP Architecture

```
+------------------+
|    MCP HOST      |  (e.g., VS Code, Claude Desktop, your app)
|  +------------+  |
|  | MCP CLIENT |  |  <-- Maintains 1:1 connection with a server
|  +-----+------+  |
+--------|--------+
         | (MCP Protocol - JSON-RPC over stdio/SSE)
+--------|--------+
|  MCP SERVER      |  (exposes tools, resources, prompts)
|  +------------+  |
|  | Tools      |  |  <-- Functions the LLM can call
|  | Resources  |  |  <-- Data the LLM can read
|  | Prompts    |  |  <-- Pre-built prompt templates
|  +------------+  |
+------------------+
```

### 1.3 MCP Primitives

| Primitive | Description | Example | Direction |
|-----------|-------------|---------|----------|
| **Tools** | Functions the LLM can execute | Calculator, DB query, API call | Client --> Server |
| **Resources** | Data/content the LLM can read | File contents, DB records | Client --> Server |
| **Prompts** | Pre-built prompt templates | Code review template, analysis prompt | Server --> Client |

### 1.4 Transport Mechanisms

| Transport | Use Case | How It Works |
|-----------|----------|-------------|
| **stdio** | Local servers | Communication via stdin/stdout |
| **SSE** | Remote servers | HTTP Server-Sent Events |

---
## Part 2: Building Your First MCP Server (Hands-On - 25 min)

We will build an MCP server that provides:
1. A calculator tool (add, multiply)
2. A weather lookup tool (mock)
3. A college info resource

**Important**: MCP servers run as separate processes. We will create the server as a Python script, then test it.

In [ ]:
# 2.1 Let's first understand the MCP Python SDK structure
# We will create the MCP server as a standalone Python file

mcp_server_code = '''
# mcp_college_server.py
# A custom MCP server for the workshop

from mcp.server.fastmcp import FastMCP
import json
from datetime import datetime

# Create an MCP server instance
mcp = FastMCP("College Assistant Server")

# ========================================
# TOOL 1: Calculator
# ========================================
@mcp.tool()
def calculator(operation: str, a: float, b: float) -> str:
    """Perform basic arithmetic operations.

    Args:
        operation: One of 'add', 'subtract', 'multiply', 'divide'
        a: First number
        b: Second number
    """
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }

    if operation not in operations:
        return f"Error: Unknown operation '{operation}'. Use: add, subtract, multiply, divide"

    result = operations[operation]
    return f"{a} {operation} {b} = {result}"


# ========================================
# TOOL 2: College Fee Calculator
# ========================================
@mcp.tool()
def calculate_total_fees(quota: str, years: int, include_hostel: bool, include_transport: bool) -> str:
    """Calculate total college fees for a student at SIT.

    Args:
        quota: One of 'government', 'comedk', 'management'
        years: Number of years (typically 4)
        include_hostel: Whether to include hostel fees
        include_transport: Whether to include transport fees
    """
    fee_structure = {
        "government": 55000,
        "comedk": 120000,
        "management": 250000
    }

    if quota not in fee_structure:
        return f"Error: Unknown quota '{quota}'. Use: government, comedk, management"

    tuition_per_year = fee_structure[quota]
    hostel_per_year = 75000 if include_hostel else 0
    transport_per_year = 15000 if include_transport else 0

    total_per_year = tuition_per_year + hostel_per_year + transport_per_year
    total = total_per_year * years

    breakdown = {
        "quota": quota,
        "tuition_per_year": tuition_per_year,
        "hostel_per_year": hostel_per_year,
        "transport_per_year": transport_per_year,
        "total_per_year": total_per_year,
        "years": years,
        "grand_total": total
    }

    return json.dumps(breakdown, indent=2)


# ========================================
# TOOL 3: Mock Weather Service
# ========================================
@mcp.tool()
def get_weather(city: str) -> str:
    """Get current weather for a city (mock data for workshop).

    Args:
        city: Name of the city
    """
    # Mock weather data
    weather_data = {
        "tumkur": {"temp": 28, "condition": "Partly Cloudy", "humidity": 65},
        "bangalore": {"temp": 24, "condition": "Clear", "humidity": 55},
        "mysore": {"temp": 26, "condition": "Sunny", "humidity": 60},
        "default": {"temp": 25, "condition": "Unknown", "humidity": 50}
    }

    city_lower = city.lower()
    data = weather_data.get(city_lower, weather_data["default"])

    return json.dumps({
        "city": city,
        "temperature_celsius": data["temp"],
        "condition": data["condition"],
        "humidity_percent": data["humidity"],
        "timestamp": datetime.now().isoformat()
    }, indent=2)


# ========================================
# TOOL 4: Student GPA Calculator
# ========================================
@mcp.tool()
def calculate_gpa(grades: str) -> str:
    """Calculate GPA from a comma-separated list of grades.

    Args:
        grades: Comma-separated grades like 'S,A,B,A,S,C'
                Grade scale: S=10, A=9, B=8, C=7, D=6, E=5, F=0
    """
    grade_points = {"S": 10, "A": 9, "B": 8, "C": 7, "D": 6, "E": 5, "F": 0}

    grade_list = [g.strip().upper() for g in grades.split(",")]

    invalid_grades = [g for g in grade_list if g not in grade_points]
    if invalid_grades:
        return f"Error: Invalid grades: {invalid_grades}. Valid: S, A, B, C, D, E, F"

    points = [grade_points[g] for g in grade_list]
    gpa = sum(points) / len(points)

    return json.dumps({
        "grades": grade_list,
        "grade_points": points,
        "gpa": round(gpa, 2),
        "total_subjects": len(grade_list),
        "classification": "First Class with Distinction" if gpa >= 7.5 else "First Class" if gpa >= 6.5 else "Second Class" if gpa >= 5.5 else "Pass" if gpa >= 4.0 else "Fail"
    }, indent=2)


# ========================================
# RESOURCE: College Information
# ========================================
@mcp.resource("college://info")
def get_college_info() -> str:
    """Get general information about the college."""
    return json.dumps({
        "name": "Sidganga Institute of Technology",
        "location": "Tumkur, Karnataka",
        "established": 1963,
        "affiliation": "VTU, Belagavi",
        "accreditation": "NAAC A Grade, NBA",
        "departments": 7,
        "placement_rate": "85%"
    }, indent=2)


# ========================================
# RESOURCE: Department List
# ========================================
@mcp.resource("college://departments")
def get_departments() -> str:
    """Get list of all departments and their seat counts."""
    departments = [
        {"name": "Computer Science and Engineering", "code": "CSE", "seats": 180},
        {"name": "Information Science and Engineering", "code": "ISE", "seats": 120},
        {"name": "Electronics and Communication", "code": "ECE", "seats": 120},
        {"name": "Mechanical Engineering", "code": "ME", "seats": 120},
        {"name": "Civil Engineering", "code": "CE", "seats": 60},
        {"name": "AI and Machine Learning", "code": "AIML", "seats": 60},
        {"name": "Data Science", "code": "DS", "seats": 60}
    ]
    return json.dumps(departments, indent=2)


# ========================================
# PROMPT: Student Advisor
# ========================================
@mcp.prompt()
def student_advisor(student_query: str) -> str:
    """Generate a prompt for advising students."""
    return f"""You are a friendly academic advisor at SIT, Tumkur.
A student has the following query: {student_query}

Provide helpful, accurate advice based on SIT policies.
Be encouraging and supportive. If you are unsure, suggest they contact the relevant office."""


if __name__ == "__main__":
    # Run the server
    mcp.run(transport="stdio")
'''

# Save the MCP server file
with open("../data/mcp_college_server.py", "w") as f:
    f.write(mcp_server_code)

print("MCP server code saved to ../data/mcp_college_server.py")
print("\nThis server exposes:")
print("  Tools: calculator, calculate_total_fees, get_weather, calculate_gpa")
print("  Resources: college://info, college://departments")
print("  Prompts: student_advisor")

### 2.2 Understanding the MCP Server Code

Let's break down the key parts:

**1. Server Creation:**
```python
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("College Assistant Server")
```
FastMCP is the high-level API for creating MCP servers. The name identifies your server.

**2. Tools (decorated with `@mcp.tool()`):**
```python
@mcp.tool()
def calculator(operation: str, a: float, b: float) -> str:
    """Docstring becomes the tool description for the LLM."""
    ...
```
- Type hints (`str`, `float`, `bool`) define the parameter schema
- Docstring describes the tool to the LLM
- Return value is always a string

**3. Resources (decorated with `@mcp.resource()`):**
```python
@mcp.resource("college://info")
def get_college_info() -> str:
    ...
```
- URI-based access pattern
- Read-only data sources

**4. Prompts (decorated with `@mcp.prompt()`):**
```python
@mcp.prompt()
def student_advisor(student_query: str) -> str:
    ...
```
- Pre-built prompt templates the client can use

In [ ]:
# 2.3 Test the MCP tools directly (without MCP protocol)
# This helps verify the logic before running as an MCP server

import json
from datetime import datetime

# Replicate the tool functions here for testing
def calculator(operation, a, b):
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }
    result = operations.get(operation, f"Unknown operation: {operation}")
    return f"{a} {operation} {b} = {result}"

def calculate_total_fees(quota, years, include_hostel, include_transport):
    fee_structure = {"government": 55000, "comedk": 120000, "management": 250000}
    tuition = fee_structure.get(quota, 0)
    hostel = 75000 if include_hostel else 0
    transport = 15000 if include_transport else 0
    total_per_year = tuition + hostel + transport
    return json.dumps({
        "quota": quota,
        "total_per_year": total_per_year,
        "years": years,
        "grand_total": total_per_year * years
    }, indent=2)

def calculate_gpa(grades_str):
    grade_points = {"S": 10, "A": 9, "B": 8, "C": 7, "D": 6, "E": 5, "F": 0}
    grades = [g.strip().upper() for g in grades_str.split(",")]
    points = [grade_points[g] for g in grades]
    gpa = sum(points) / len(points)
    return json.dumps({"grades": grades, "gpa": round(gpa, 2)}, indent=2)

# Test each tool
print("=== Calculator ===")
print(calculator("add", 15, 27))
print(calculator("multiply", 6, 7))
print()

print("=== Fee Calculator ===")
print(calculate_total_fees("government", 4, True, True))
print()

print("=== GPA Calculator ===")
print(calculate_gpa("S,A,B,A,S,C,A,B"))

---
## Part 3: Running and Testing the MCP Server (Hands-On - 15 min)

### 3.1 Running the MCP Server

The MCP server runs as a separate process. To test it:

**Option 1: Run directly**
```bash
cd workshop/data
python mcp_college_server.py
```

**Option 2: Use with VS Code / Claude Desktop**
Add to your MCP configuration:
```json
{
    "mcpServers": {
        "college-assistant": {
            "command": "python",
            "args": ["path/to/mcp_college_server.py"]
        }
    }
}
```

**Option 3: Use the MCP Inspector** (recommended for testing)
```bash
npx @modelcontextprotocol/inspector python mcp_college_server.py
```

In [ ]:
# 3.2 Testing MCP Server programmatically using subprocess
# This simulates how an MCP client communicates with the server via stdio

import subprocess
import json

# The MCP protocol uses JSON-RPC 2.0 over stdio
# Here is what a tool call looks like in the MCP protocol:

sample_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "calculator",
        "arguments": {
            "operation": "multiply",
            "a": 15,
            "b": 4
        }
    }
}

print("=== Sample MCP JSON-RPC Request ===")
print(json.dumps(sample_request, indent=2))
print()
print("This is how an MCP client communicates with the server.")
print("The server reads this from stdin, processes it, and writes the response to stdout.")
print()
print("In practice, you would use the MCP client library to handle this automatically.")
print("We will build a full MCP client in Day 2, Session 6!")

In [ ]:
# 3.3 MCP Configuration for VS Code (for reference)

vscode_mcp_config = {
    "mcpServers": {
        "college-assistant": {
            "command": "python",
            "args": ["workshop/data/mcp_college_server.py"],
            "env": {}
        }
    }
}

print("=== VS Code MCP Configuration ===")
print("Save this in .vscode/mcp.json or VS Code settings:")
print()
print(json.dumps(vscode_mcp_config, indent=2))
print()
print("After adding this configuration:")
print("1. Restart VS Code")
print("2. Open Copilot Chat")
print("3. The tools will be available for Copilot to use")

---
## Part 4: How MCP Fits into the AI Ecosystem (Theory - 5 min)

### MCP vs Function Calling vs LangChain Tools

| Feature | LangChain Tools | OpenAI Function Calling | MCP |
|---------|----------------|------------------------|-----|
| **Standard** | Framework-specific | Provider-specific | Open protocol |
| **Interop** | Only within LangChain | Only with OpenAI-compatible | Any MCP client |
| **Discovery** | Manual registration | Manual schema | Dynamic discovery |
| **Transport** | In-process | HTTP API | stdio / SSE |
| **Reusability** | Must rewrite for each app | Must rewrite | Write once, use everywhere |

### MCP's Unique Value:
1. **Write Once, Use Everywhere**: One MCP server works with Claude, VS Code, any MCP client
2. **Dynamic Discovery**: Clients discover available tools at runtime
3. **Security**: Built-in permission model (human-in-the-loop for sensitive tools)
4. **Ecosystem**: Growing library of pre-built MCP servers (GitHub, Slack, databases, etc.)

### Where MCP is Used Today:
- **Claude Desktop**: Connect Claude to local files, databases, APIs
- **VS Code Copilot**: Extend Copilot with custom tools
- **Custom AI Apps**: Build production AI systems with standardized tool access
- **Enterprise**: Secure, auditable AI-to-tool connections

---

## Summary

### Key Takeaways:
1. **MCP standardizes** how LLMs connect to external tools and data (like USB for AI)
2. **Architecture**: Host (app) contains Client, which connects to Server (tools)
3. **Three primitives**: Tools (actions), Resources (data), Prompts (templates)
4. **FastMCP** makes it easy to build MCP servers with Python decorators
5. MCP servers run as **separate processes** communicating via stdio or SSE
6. MCP enables **write-once, use-everywhere** tool integration

### Next (Day 2): MCP Deep Dive
- Build an MCP client
- Connect LangChain agents to MCP servers
- Run MCP servers in Docker containers
- Build production-ready MCP integrations